# Notebook 03: Expansion Signals
## SaaS Expansion Analytics — RavenStack Dataset
**Objective:** Identify which behaviours and characteristics predict upgrades — feature usage, support satisfaction, seat growth, and account profile. These become the features for the ML model in Notebook 04.

In [1]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.2f}'.format)

print("Libraries loaded")

Libraries loaded


In [2]:
accounts      = pd.read_csv('../data/raw/ravenstack_accounts.csv')
subscriptions = pd.read_csv('../data/raw/ravenstack_subscriptions.csv')
feature_usage = pd.read_csv('../data/raw/ravenstack_feature_usage.csv')
support       = pd.read_csv('../data/raw/ravenstack_support_tickets.csv')

# Parse dates
accounts['signup_date']         = pd.to_datetime(accounts['signup_date'])
subscriptions['start_date']     = pd.to_datetime(subscriptions['start_date'])
subscriptions['end_date']       = pd.to_datetime(subscriptions['end_date'])
feature_usage['usage_date']     = pd.to_datetime(feature_usage['usage_date'])
support['submitted_at']         = pd.to_datetime(support['submitted_at'])

# Active subscriptions only, exclude $0 MRR
active_subs = subscriptions[
    (subscriptions['end_date'].isna()) &
    (subscriptions['mrr_amount'] > 0)
]

print(f"Active subscriptions: {len(active_subs):,}")
print(f"Accounts:             {len(accounts):,}")
print(f"Feature usage events: {len(feature_usage):,}")
print(f"Support tickets:      {len(support):,}")

Active subscriptions: 3,814
Accounts:             500
Feature usage events: 25,000
Support tickets:      2,000


## 1. Build Account-Level Feature Table
Aggregate all signals to account level — one row per account.
This is the foundation for the ML model.

In [3]:
# ── Subscription signals ──────────────────────────────────────
sub_features = (active_subs
    .groupby('account_id')
    .agg(
        total_mrr        =('mrr_amount', 'sum'),
        avg_mrr          =('mrr_amount', 'mean'),
        max_mrr          =('mrr_amount', 'max'),
        total_seats      =('seats', 'sum'),
        avg_seats        =('seats', 'mean'),
        n_subscriptions  =('subscription_id', 'count'),
        has_upgraded     =('upgrade_flag', 'max'),
        has_downgraded   =('downgrade_flag', 'max'),
        is_annual        =('billing_frequency',
                           lambda x: (x == 'annual').any()),
        auto_renew       =('auto_renew_flag', 'max')
    )
    .reset_index()
)

# ── Feature usage signals ─────────────────────────────────────
# Join feature_usage → subscriptions → account
usage_with_account = feature_usage.merge(
    active_subs[['subscription_id', 'account_id']],
    on='subscription_id',
    how='inner'
)

usage_features = (usage_with_account
    .groupby('account_id')
    .agg(
        total_usage_count    =('usage_count', 'sum'),
        avg_usage_count      =('usage_count', 'mean'),
        total_usage_duration =('usage_duration_secs', 'sum'),
        unique_features_used =('feature_name', 'nunique'),
        total_errors         =('error_count', 'sum'),
        beta_feature_usage   =('is_beta_feature', 'sum')
    )
    .reset_index()
)

# ── Support signals ───────────────────────────────────────────
support_features = (support
    .groupby('account_id')
    .agg(
        n_tickets            =('ticket_id', 'count'),
        avg_satisfaction     =('satisfaction_score', 'mean'),
        avg_resolution_hours =('resolution_time_hours', 'mean'),
        n_escalations        =('escalation_flag', 'sum'),
        n_urgent             =('priority',
                               lambda x: (x == 'urgent').sum())
    )
    .reset_index()
)

print("Feature tables built:")
print(f"  sub_features:     {sub_features.shape}")
print(f"  usage_features:   {usage_features.shape}")
print(f"  support_features: {support_features.shape}")

Feature tables built:
  sub_features:     (500, 11)
  usage_features:   (500, 7)
  support_features: (492, 6)


In [4]:
# Master feature table — one row per account
df = (accounts
    .merge(sub_features,     on='account_id', how='left')
    .merge(usage_features,   on='account_id', how='left')
    .merge(support_features, on='account_id', how='left')
)

# Fill nulls — accounts with no usage/support data
df['total_usage_count']    = df['total_usage_count'].fillna(0)
df['unique_features_used'] = df['unique_features_used'].fillna(0)
df['total_errors']         = df['total_errors'].fillna(0)
df['beta_feature_usage']   = df['beta_feature_usage'].fillna(0)
df['n_tickets']            = df['n_tickets'].fillna(0)
df['avg_satisfaction']     = df['avg_satisfaction'].fillna(
    df['avg_satisfaction'].median()
)
df['n_escalations']        = df['n_escalations'].fillna(0)
df['n_urgent']             = df['n_urgent'].fillna(0)
df['avg_resolution_hours'] = df['avg_resolution_hours'].fillna(
    df['avg_resolution_hours'].median()
)

print(f"Master feature table: {df.shape}")
print(f"Columns: {list(df.columns)}")
print(f"\nNulls remaining:")
print(df.isnull().sum()[df.isnull().sum() > 0])

Master feature table: (500, 31)
Columns: ['account_id', 'account_name', 'industry', 'country', 'signup_date', 'referral_source', 'plan_tier', 'seats', 'is_trial', 'churn_flag', 'total_mrr', 'avg_mrr', 'max_mrr', 'total_seats', 'avg_seats', 'n_subscriptions', 'has_upgraded', 'has_downgraded', 'is_annual', 'auto_renew', 'total_usage_count', 'avg_usage_count', 'total_usage_duration', 'unique_features_used', 'total_errors', 'beta_feature_usage', 'n_tickets', 'avg_satisfaction', 'avg_resolution_hours', 'n_escalations', 'n_urgent']

Nulls remaining:
Series([], dtype: int64)


## 2. Feature Usage vs Upgrade
Do accounts that use more features upgrade more?

In [5]:
# Compare usage between upgraded vs non-upgraded accounts
upgraded     = df[df['has_upgraded'] == True]
not_upgraded = df[df['has_upgraded'] == False]

metrics = {
    'Avg Usage Count':      ('total_usage_count', 'mean'),
    'Unique Features Used': ('unique_features_used', 'mean'),
    'Avg Usage Duration (s)':('total_usage_duration', 'mean'),
    'Beta Feature Usage':   ('beta_feature_usage', 'mean'),
    'Avg Errors':           ('total_errors', 'mean')
}

print("=== Feature Usage: Upgraded vs Not Upgraded ===\n")
print(f"{'Metric':<30} {'Upgraded':>12} {'Not Upgraded':>14} {'Diff':>8}")
print("─" * 68)

for label, (col, agg) in metrics.items():
    val_up  = upgraded[col].mean()
    val_not = not_upgraded[col].mean()
    diff    = ((val_up - val_not) / val_not * 100)
    print(f"{label:<30} {val_up:>12.1f} {val_not:>14.1f} "
          f"{diff:>+7.1f}%")

=== Feature Usage: Upgraded vs Not Upgraded ===

Metric                             Upgraded   Not Upgraded     Diff
────────────────────────────────────────────────────────────────────
Avg Usage Count                       418.2          344.1   +21.5%
Unique Features Used                   25.2           22.1   +14.2%
Avg Usage Duration (s)             127419.9       103765.9   +22.8%
Beta Feature Usage                      4.3            3.4   +24.8%
Avg Errors                             23.6           19.1   +23.6%


In [6]:
fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=[
        'Unique Features Used',
        'Total Usage Count'
    ]
)

for col, title, row, c in [
    ('unique_features_used', 'Unique Features', 1, 1),
    ('total_usage_count',    'Total Usage',     1, 2)
]:
    fig.add_trace(go.Box(
        y=upgraded[col],
        name='Upgraded',
        marker_color='#2ecc71',
        showlegend=(c == 1)
    ), row=1, col=c)

    fig.add_trace(go.Box(
        y=not_upgraded[col],
        name='Not Upgraded',
        marker_color='#e74c3c',
        showlegend=(c == 1)
    ), row=1, col=c)

fig.update_layout(
    height=400,
    font=dict(size=13),
    title='Feature Usage Distribution: Upgraded vs Not Upgraded',
    boxmode='group'
)
fig.show()

## 3. Support Satisfaction vs Upgrade
Do happier customers upgrade more?

In [7]:
print("=== Support Signals: Upgraded vs Not Upgraded ===\n")
print(f"{'Metric':<30} {'Upgraded':>12} {'Not Upgraded':>14} {'Diff':>8}")
print("─" * 68)

support_metrics = {
    'Avg Satisfaction Score': ('avg_satisfaction', 'mean'),
    'Avg Tickets':            ('n_tickets', 'mean'),
    'Avg Escalations':        ('n_escalations', 'mean'),
    'Avg Resolution Hours':   ('avg_resolution_hours', 'mean'),
    'Avg Urgent Tickets':     ('n_urgent', 'mean')
}

for label, (col, agg) in support_metrics.items():
    val_up  = upgraded[col].mean()
    val_not = not_upgraded[col].mean()
    diff    = ((val_up - val_not) / val_not * 100)
    print(f"{label:<30} {val_up:>12.2f} {val_not:>14.2f} "
          f"{diff:>+7.1f}%")

=== Support Signals: Upgraded vs Not Upgraded ===

Metric                             Upgraded   Not Upgraded     Diff
────────────────────────────────────────────────────────────────────
Avg Satisfaction Score                 3.98           3.95    +0.9%
Avg Tickets                            3.93           4.08    -3.7%
Avg Escalations                        0.21           0.17   +23.3%
Avg Resolution Hours                  36.77          35.61    +3.3%
Avg Urgent Tickets                     1.03           1.03    -0.4%


## 4. Account Profile vs Upgrade
Do certain industries, plans or referral sources upgrade more?

In [8]:
fig = make_subplots(
    rows=1, cols=3,
    subplot_titles=[
        'Upgrade Rate by Industry',
        'Upgrade Rate by Plan Tier',
        'Upgrade Rate by Referral'
    ]
)

for col, c in [
    ('industry', 1),
    ('plan_tier', 2),
    ('referral_source', 3)
]:
    rates = (df.groupby(col)['has_upgraded']
             .mean()
             .mul(100)
             .round(1)
             .sort_values()
             .reset_index())
    rates.columns = [col, 'upgrade_rate']

    max_val = rates['upgrade_rate'].max()

    fig.add_trace(go.Bar(
        x=rates['upgrade_rate'],
        y=rates[col],
        orientation='h',
        text=[f"{x:.1f}%" for x in rates['upgrade_rate']],
        textposition='outside',
        marker_color='#3498db',
        showlegend=False
    ), row=1, col=c)

    fig.update_xaxes(
        range=[0, max_val * 1.3],
        row=1, col=c
    )

fig.update_layout(
    height=450,
    font=dict(size=12),
    title='Upgrade Rate by Account Profile'
)
fig.show()

## 5. Seat Growth vs Upgrade
Accounts adding seats signal growth — do they upgrade more?

In [9]:
print("=== Seat Analysis: Upgraded vs Not Upgraded ===\n")
print(f"{'Metric':<25} {'Upgraded':>12} {'Not Upgraded':>14} {'Diff':>8}")
print("─" * 63)

seat_metrics = {
    'Avg Total Seats':  'total_seats',
    'Avg MRR':          'total_mrr',
    'Annual Billing %': 'is_annual',
    'Auto Renew %':     'auto_renew'
}

for label, col in seat_metrics.items():
    val_up  = upgraded[col].mean()
    val_not = not_upgraded[col].mean()
    diff    = ((val_up - val_not) / val_not * 100)
    print(f"{label:<25} {val_up:>12.2f} {val_not:>14.2f} "
          f"{diff:>+7.1f}%")

# Seat distribution chart
fig = go.Figure()
fig.add_trace(go.Box(
    y=upgraded['total_seats'],
    name='Upgraded',
    marker_color='#2ecc71'
))
fig.add_trace(go.Box(
    y=not_upgraded['total_seats'],
    name='Not Upgraded',
    marker_color='#e74c3c'
))
fig.update_layout(
    title='Seat Distribution: Upgraded vs Not Upgraded',
    height=400,
    font=dict(size=13),
    yaxis_title='Total Seats'
)
fig.show()

=== Seat Analysis: Upgraded vs Not Upgraded ===

Metric                        Upgraded   Not Upgraded     Diff
───────────────────────────────────────────────────────────────
Avg Total Seats                 247.96         202.21   +22.6%
Avg MRR                       22430.25       17860.91   +25.6%
Annual Billing %                  0.99           0.97    +1.6%
Auto Renew %                      1.00           1.00    +0.0%


## 6. Correlation Matrix
Which features correlate most strongly with upgrade behaviour?

In [10]:
numeric_cols = [
    'total_mrr', 'total_seats', 'total_usage_count',
    'unique_features_used', 'total_usage_duration',
    'beta_feature_usage', 'total_errors',
    'n_tickets', 'avg_satisfaction', 'n_escalations',
    'avg_resolution_hours', 'has_upgraded'
]

corr = df[numeric_cols].corr()['has_upgraded'].drop('has_upgraded')
corr_df = corr.abs().sort_values(ascending=True).reset_index()
corr_df.columns = ['feature', 'correlation']

fig = go.Figure(go.Bar(
    x=corr_df['correlation'],
    y=corr_df['feature'],
    orientation='h',
    marker_color=[
        '#2ecc71' if x > 0.1 else '#3498db'
        for x in corr_df['correlation']
    ],
    text=[f"{x:.3f}" for x in corr_df['correlation']],
    textposition='outside'
))

max_corr = corr_df['correlation'].max()
fig.update_layout(
    title='Feature Correlation with Upgrade (absolute value)',
    height=450,
    font=dict(size=13),
    xaxis=dict(range=[0, max_corr * 1.3]),
    xaxis_title='Absolute Correlation'
)
fig.show()

print("\nTop 5 features correlated with upgrade:")
print(corr_df.tail(5).sort_values(
    'correlation', ascending=False
).to_string(index=False))


Top 5 features correlated with upgrade:
             feature  correlation
total_usage_duration         0.25
   total_usage_count         0.24
unique_features_used         0.24
        total_errors         0.21
  beta_feature_usage         0.18


In [11]:
df.to_csv('../data/processed/account_features.csv', index=False)
print(f"Saved account_features.csv — {df.shape[0]} accounts, "
      f"{df.shape[1]} features")

Saved account_features.csv — 500 accounts, 31 features


## 7. Key Findings

**Feature usage is the strongest expansion signal:**
- Upgraded accounts use the product 21.5% more frequently
- They use 14.2% more unique features — broader adoption
- Usage duration is 22.8% higher — deeper engagement
- Beta feature usage is 24.8% higher — power user behaviour
- Top 5 correlations with upgrade are all usage-based (0.18-0.25)

**Seat count and MRR are strong secondary signals:**
- Upgraded accounts have 22.6% more seats — company growth proxy
- Upgraded accounts have 25.6% higher MRR — already more invested

**Support satisfaction is a weak signal:**
- Satisfaction score virtually identical (3.98 vs 3.95)
- Ticket volume slightly lower for upgraded accounts
- Support health does not predict expansion in this dataset

**Implications for the ML model (Notebook 04):**
- Feature usage metrics will likely dominate feature importance
- Seat count and MRR are strong supporting features
- Support features may add marginal value
- upgrade_flag is the target variable